# Hierarchical Indexing

Hierarchical / Multi-level indexing is very exciting as it opens the door to some quite sophisticated data analysis and manipulation, especially for working with higher dimensional data. In essence, it enables you to store and manipulate data with an arbitrary number of dimensions in lower dimensional data structures like Series (1d) and DataFrame (2d).

## Creating a MultiIndex (hierarchical index) object

The MultiIndex object is the hierarchical analogue of the standard Index object which typically stores the axis labels in pandas objects. You can think of MultiIndex as an array of tuples where each tuple is unique. A MultiIndex can be created from a list of arrays (using MultiIndex.from_arrays()), an array of tuples (using MultiIndex.from_tuples()), a crossed set of iterables (using MultiIndex.from_product()), or a DataFrame (using MultiIndex.from_frame()). The Index constructor will attempt to return a MultiIndex when it is passed a list of tuples. The following examples demonstrate different ways to initialize MultiIndexes.

In [2]:
import pandas as pd
import numpy as np

In [3]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]
arrays

[['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux'],
 ['one', 'two', 'one', 'two', 'one', 'two', 'one', 'two']]

In [4]:
arrays[0]

['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux']

In [5]:
tuples = list(zip(*arrays))
tuples1 = list(zip(arrays[0],arrays[1]))
print(f'{tuples == tuples1}\n')
tuples

True



[('bar', 'one'),
 ('bar', 'two'),
 ('baz', 'one'),
 ('baz', 'two'),
 ('foo', 'one'),
 ('foo', 'two'),
 ('qux', 'one'),
 ('qux', 'two')]

In [10]:
languages = ['Java', 'Python', 'JavaScript']
versions = [14, 3, 6]

result = zip(languages, versions)
list(result)

[('Java', 14), ('Python', 3), ('JavaScript', 6)]

In [11]:
index = pd.MultiIndex.from_tuples(tuples, names=["first", "second"])
index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [12]:
s = pd.Series(np.random.randn(8), index=index)
s

first  second
bar    one       0.320412
       two       0.057621
baz    one      -0.812672
       two      -2.332426
foo    one      -1.012450
       two       0.935820
qux    one      -0.754974
       two       0.917404
dtype: float64

paring every elements in two iterables (모든 경우의 수)

In [13]:
iterables = [["bar", "baz", "foo", "qux"], ["one", "two"]]

In [14]:
pd.MultiIndex.from_product(iterables, names=["first", "second"])

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [15]:
df = pd.DataFrame(
    [["bar", "one"], ["bar", "two"], ["foo", "one"], ["foo", "two"]],
    columns=["first", "second"],
)
df

,first,second
0,bar,one
1,bar,two
2,foo,one
3,foo,two


In [16]:
pd.MultiIndex.from_frame(df)

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('foo', 'one'),
            ('foo', 'two')],
           names=['first', 'second'])

As a convenience, you can pass a list of arrays directly into Series or DataFrame to construct a MultiIndex automatically:

In [17]:
arrays = [
    np.array(["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"]),
    np.array(["one", "two", "one", "two", "one", "two", "one", "two"]),
]
s = pd.Series(np.random.randn(8), index=arrays)
s

bar  one   -1.229143
     two   -0.988839
baz  one   -0.419775
     two    0.233544
foo  one    0.698177
     two   -2.655142
qux  one   -0.485734
     two   -0.168731
dtype: float64

## Reconstructing the level labels

In [18]:
index.get_level_values(1) # by ind

Index(['one', 'two', 'one', 'two', 'one', 'two', 'one', 'two'], dtype='object', name='second')

In [19]:
index.get_level_values("first") # by specific name

Index(['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux'], dtype='object', name='first')

## Basic indexing on axis with MultiIndex

One of the important features of hierarchical indexing is that you can select data by a “partial” label identifying a subgroup in the data. Partial selection “drops” levels of the hierarchical index in the result in a completely analogous way to selecting a column in a regular DataFrame:

In [20]:
df = pd.DataFrame(np.random.randn(3, 8), index=["A", "B", "C"], columns=index)
df

first        bar                 baz                 foo                 qux  \
second       one       two       one       two       one       two       one   
A       1.398387 -0.096010  0.581279  0.275705 -0.891497 -0.826040  0.516785   
B       0.901099  1.526311 -0.253146 -0.381913 -0.388298 -1.112424  0.392079   
C      -1.337388  0.904940  1.420054 -1.221479 -1.190872  0.285776 -0.886939   

first             
second       two  
A      -0.289234  
B       0.255499  
C      -0.008080

In [21]:
df[['bar','baz']]

first        bar                 baz          
second       one       two       one       two
A       1.398387 -0.096010  0.581279  0.275705
B       0.901099  1.526311 -0.253146 -0.381913
C      -1.337388  0.904940  1.420054 -1.221479

In [22]:
df["bar","one"]

A    1.398387
B    0.901099
C   -1.337388
Name: (bar, one), dtype: float64

## Define levels

In [23]:
df.columns.levels

FrozenList([['bar', 'baz', 'foo', 'qux'], ['one', 'two']])

In [24]:
df[["foo","qux"]].columns.levels  # same as original

FrozenList([['bar', 'baz', 'foo', 'qux'], ['one', 'two']])

You can use instead

In [25]:
df[["foo", "qux"]].columns.to_numpy()

array([('foo', 'one'), ('foo', 'two'), ('qux', 'one'), ('qux', 'two')],
      dtype=object)

In [26]:
df[["foo", "qux"]].columns.get_level_values(0)

Index(['foo', 'foo', 'qux', 'qux'], dtype='object', name='first')

In [27]:
df[["foo", "qux"]].columns.get_level_values(1)

Index(['one', 'two', 'one', 'two'], dtype='object', name='second')

## Data alignment and using reindex

Operations between differently-indexed objects having MultiIndex on the axes will work as you expect; data alignment will work the same as an Index of tuples:

In [28]:
s

bar  one   -1.229143
     two   -0.988839
baz  one   -0.419775
     two    0.233544
foo  one    0.698177
     two   -2.655142
qux  one   -0.485734
     two   -0.168731
dtype: float64

In [29]:
s[:-2]

bar  one   -1.229143
     two   -0.988839
baz  one   -0.419775
     two    0.233544
foo  one    0.698177
     two   -2.655142
dtype: float64

In [30]:
s[3:]

baz  two    0.233544
foo  one    0.698177
     two   -2.655142
qux  one   -0.485734
     two   -0.168731
dtype: float64

In [31]:
s

bar  one   -1.229143
     two   -0.988839
baz  one   -0.419775
     two    0.233544
foo  one    0.698177
     two   -2.655142
qux  one   -0.485734
     two   -0.168731
dtype: float64

In [32]:
s + s[:-2] # operating based on tuple(multi) indexes

bar  one   -2.458286
     two   -1.977678
baz  one   -0.839550
     two    0.467089
foo  one    1.396354
     two   -5.310284
qux  one         NaN
     two         NaN
dtype: float64

In [33]:
s[::2] # step

bar  one   -1.229143
baz  one   -0.419775
foo  one    0.698177
qux  one   -0.485734
dtype: float64

In [34]:
s + s[::2]

bar  one   -2.458286
     two         NaN
baz  one   -0.839550
     two         NaN
foo  one    1.396354
     two         NaN
qux  one   -0.971469
     two         NaN
dtype: float64

In [35]:
index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [36]:
s[index[:3]]

bar  one   -1.229143
     two   -0.988839
baz  one   -0.419775
dtype: float64

The reindex() method of Series/DataFrames can be called with another MultiIndex, or even a list or array of tuples:

In [37]:
np.random.randn(8).shape

(8,)

In [38]:
s

bar  one   -1.229143
     two   -0.988839
baz  one   -0.419775
     two    0.233544
foo  one    0.698177
     two   -2.655142
qux  one   -0.485734
     two   -0.168731
dtype: float64

In [39]:
s.index[-1]

('qux', 'two')

In [40]:
s.reindex(s.index[::-1])

qux  two   -0.168731
     one   -0.485734
foo  two   -2.655142
     one    0.698177
baz  two    0.233544
     one   -0.419775
bar  two   -0.988839
     one   -1.229143
dtype: float64

In [41]:
s.reindex(index[:3])

first  second
bar    one      -1.229143
       two      -0.988839
baz    one      -0.419775
dtype: float64

## Advanced indexing with hierarchical index

In [42]:
df

first        bar                 baz                 foo                 qux  \
second       one       two       one       two       one       two       one   
A       1.398387 -0.096010  0.581279  0.275705 -0.891497 -0.826040  0.516785   
B       0.901099  1.526311 -0.253146 -0.381913 -0.388298 -1.112424  0.392079   
C      -1.337388  0.904940  1.420054 -1.221479 -1.190872  0.285776 -0.886939   

first             
second       two  
A      -0.289234  
B       0.255499  
C      -0.008080

In [43]:
df.T

A         B         C
first second                              
bar   one     1.398387  0.901099 -1.337388
      two    -0.096010  1.526311  0.904940
baz   one     0.581279 -0.253146  1.420054
      two     0.275705 -0.381913 -1.221479
foo   one    -0.891497 -0.388298 -1.190872
      two    -0.826040 -1.112424  0.285776
qux   one     0.516785  0.392079 -0.886939
      two    -0.289234  0.255499 -0.008080

In [44]:
df = df.T

In [45]:
df.index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [46]:
df[['A','B']]

A         B
first second                    
bar   one     1.398387  0.901099
      two    -0.096010  1.526311
baz   one     0.581279 -0.253146
      two     0.275705 -0.381913
foo   one    -0.891497 -0.388298
      two    -0.826040 -1.112424
qux   one     0.516785  0.392079
      two    -0.289234  0.255499

In [47]:
df.loc[("bar", "two")]

A   -0.096010
B    1.526311
C    0.904940
Name: (bar, two), dtype: float64

In [48]:
df.loc[("bar", "two"), "A"]

np.float64(-0.0960103360885342)

In [49]:
df.loc["bar"]

,A,B,C
second,,,
one,1.398387,0.901099,-1.337388
two,-0.096010,1.526311,0.904940


### partial slicing 

In [50]:
df.loc["baz":"foo"]

A         B         C
first second                              
baz   one     0.581279 -0.253146  1.420054
      two     0.275705 -0.381913 -1.221479
foo   one    -0.891497 -0.388298 -1.190872
      two    -0.826040 -1.112424  0.285776

In [51]:
df.iloc[0:4]

A         B         C
first second                              
bar   one     1.398387  0.901099 -1.337388
      two    -0.096010  1.526311  0.904940
baz   one     0.581279 -0.253146  1.420054
      two     0.275705 -0.381913 -1.221479

slicing with tuples 

In [73]:
display(df)
df.loc['one':'two']

A         B         C
first second                              
bar   one     1.398387  0.901099 -1.337388
      two    -0.096010  1.526311  0.904940
baz   one     0.581279 -0.253146  1.420054
      two     0.275705 -0.381913 -1.221479
foo   one    -0.891497 -0.388298 -1.190872
      two    -0.826040 -1.112424  0.285776
qux   one     0.516785  0.392079 -0.886939
      two    -0.289234  0.255499 -0.008080

A         B         C
first second                              
qux   one     0.516785  0.392079 -0.886939
      two    -0.289234  0.255499 -0.008080

In [52]:
df.loc[("baz", "two"):("qux", "one")]

A         B         C
first second                              
baz   two     0.275705 -0.381913 -1.221479
foo   one    -0.891497 -0.388298 -1.190872
      two    -0.826040 -1.112424  0.285776
qux   one     0.516785  0.392079 -0.886939

###  Slicing (extra)

- Design for slicing by index usaully doesn't include the spcified number
- However, calling with the specific index num, it includes the data (For practical usage)

In [53]:
s = pd.Series(np.random.randn(6), index=list("abcdef"))
s

a    0.283335
b   -0.568734
c    0.859461
d    1.139223
e   -0.340705
f    0.501160
dtype: float64

In [54]:
s.iloc[:3]
s.loc[:'d']

a    0.283335
b   -0.568734
c    0.859461
d    1.139223
dtype: float64

In [55]:
s[:"d"]

a    0.283335
b   -0.568734
c    0.859461
d    1.139223
dtype: float64

This is most definitely a “practicality beats purity” sort of thing, but it is something to watch out for if you expect label-based slicing to behave exactly in the way that standard Python integer slicing works.

In [56]:
!pwd

'pwd'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.
